# Multi-Agent Graphs

This notebook covers two common LangGraph multi-agent patterns:

- a supervisor that routes work to Researcher, Writer, and QA agents
- parallel execution with the Send API
- shared state with reducers versus worker-scoped state

LangGraph’s multi-agent docs describe supervisor-style orchestration and agent collaboration, and the graph API docs show that parallel branches can be built with the Send API. The reducer docs explain how annotated state keys are merged when parallel nodes update the same channel. 


## Learning goals

By the end of this notebook, you should be able to:

1. Build a small supervisor-led multi-agent graph.
2. Run Researcher, Writer, and QA workers in parallel.
3. Use `Send` to fan out to workers.
4. Use a reducer to collect shared outputs.
5. See the difference between shared state and worker-scoped state.


## 1) Install packages


In [1]:
%pip install -qU langgraph


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-tests 1.1.4 requires pytest<9.0.0,>=7.0.0, but you have pytest 9.0.3 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2) Import the LangGraph pieces


In [2]:
import operator
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send


## 3) Shared state versus worker state

In LangGraph, each key in the state has its own reducer. If you annotate a key with `operator.add`, multiple parallel workers can append to the same list instead of overwriting it. The use-graph-api docs show this exact pattern for parallel nodes and note that nodes in the same superstep can run concurrently.


In [3]:
class OverallState(TypedDict):
    topic: str
    plan: list[str]
    notes: Annotated[list[str], operator.add]
    draft: str
    qa_report: str
    final_report: str

class WorkerState(TypedDict):
    topic: str
    section: str
    notes: Annotated[list[str], operator.add]


## 4) Define the worker nodes

These are simple Python functions that stand in for Researcher, Writer, and QA agents.


In [4]:
def researcher(state: WorkerState):
    topic = state['topic']
    section = state['section']
    return {
        'notes': [f'Researcher gathered facts for {topic} under section: {section}'],
    }

def writer(state: WorkerState):
    topic = state['topic']
    section = state['section']
    return {
        'notes': [f'Writer drafted a clear summary for {topic} under section: {section}'],
    }

def qa(state: WorkerState):
    topic = state['topic']
    section = state['section']
    return {
        'notes': [f'QA checked the draft for {topic} under section: {section}'],
    }


## 5) Supervisor node

The supervisor decides which work needs to happen. In this notebook it creates a small plan that is then routed to the workers.

This mirrors the supervisor-style orchestration described in the LangGraph multi-agent docs.


In [5]:
def supervisor(state: OverallState):
    topic = state['topic']
    if 'report' in topic.lower() or 'analysis' in topic.lower():
        plan = ['research', 'write', 'qa']
    else:
        plan = ['research', 'qa']
    return {'plan': plan}


## 6) Parallel execution with Send

LangGraph supports returning `Send` objects from conditional edges to dynamically create worker inputs. The graph API docs and use-graph-api docs show this map-reduce style fan-out pattern, and they note that the worker outputs are merged back into shared state using reducers. 


In [6]:
def fan_out_to_workers(state: OverallState):
    sends = []
    for step in state['plan']:
        if step == 'research':
            sends.append(Send('researcher', {'topic': state['topic'], 'section': 'research', 'notes': []}))
        elif step == 'write':
            sends.append(Send('writer', {'topic': state['topic'], 'section': 'writing', 'notes': []}))
        elif step == 'qa':
            sends.append(Send('qa', {'topic': state['topic'], 'section': 'quality assurance', 'notes': []}))
    return sends

## 7) Build the graph

The graph uses a supervisor step, then fans out to workers, and finally synthesizes a report from the shared notes.


In [7]:
def synthesize(state: OverallState):
    notes_text = '\n'.join(f'- {note}' for note in state['notes'])
    return {
        'final_report': f"Topic: {state['topic']}\n\nCollected notes:\n{notes_text}",
    }

builder = StateGraph(OverallState)

builder.add_node('supervisor', supervisor)
builder.add_node('researcher', researcher)
builder.add_node('writer', writer)
builder.add_node('qa', qa)
builder.add_node('synthesize', synthesize)

builder.add_edge(START, 'supervisor')
builder.add_conditional_edges('supervisor', fan_out_to_workers)
builder.add_edge('researcher', 'synthesize')
builder.add_edge('writer', 'synthesize')
builder.add_edge('qa', 'synthesize')
builder.add_edge('synthesize', END)

graph = builder.compile()
print('Graph compiled.')


Graph compiled.


## 8) Run the graph

Because `notes` is annotated with `operator.add`, parallel workers can contribute to the same shared list. The reducer docs show that this is the correct way to merge state updates from parallel branches.


In [8]:
result = graph.invoke({
    'topic': 'Build a RAG project report',
    'plan': [],
    'notes': [],
    'draft': '',
    'qa_report': '',
    'final_report': '',
})

result


{'topic': 'Build a RAG project report',
 'plan': ['research', 'write', 'qa'],
 'notes': ['Researcher gathered facts for Build a RAG project report under section: research',
  'Writer drafted a clear summary for Build a RAG project report under section: writing',
  'QA checked the draft for Build a RAG project report under section: quality assurance'],
 'draft': '',
 'qa_report': '',
 'final_report': 'Topic: Build a RAG project report\n\nCollected notes:\n- Researcher gathered facts for Build a RAG project report under section: research\n- Writer drafted a clear summary for Build a RAG project report under section: writing\n- QA checked the draft for Build a RAG project report under section: quality assurance'}

## 9) Try a second topic


In [9]:
result_2 = graph.invoke({
    'topic': 'Short internal summary',
    'plan': [],
    'notes': [],
    'draft': '',
    'qa_report': '',
    'final_report': '',
})

result_2


{'topic': 'Short internal summary',
 'plan': ['research', 'qa'],
 'notes': ['Researcher gathered facts for Short internal summary under section: research',
  'QA checked the draft for Short internal summary under section: quality assurance'],
 'draft': '',
 'qa_report': '',
 'final_report': 'Topic: Short internal summary\n\nCollected notes:\n- Researcher gathered facts for Short internal summary under section: research\n- QA checked the draft for Short internal summary under section: quality assurance'}

## 10) Shared state versus scoped state

Shared state is the reducer-backed data all branches can write to, like `notes`.

Scoped state is the per-worker input passed through `Send`, like the worker’s own `section` field. That allows each branch to receive a tailored input while still contributing back to the shared result.


In [10]:
worker_example = {'topic': 'Demo', 'section': 'research', 'notes': []}
researcher(worker_example)
writer(worker_example)
qa(worker_example)


{'notes': ['QA checked the draft for Demo under section: research']}

## 11) Visualize the graph

The LangGraph docs show that you can inspect the graph structure after compilation using the graph object’s visualization utilities. 


In [11]:
mermaid = graph.get_graph().draw_mermaid()
print(mermaid)


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__(<p>__start__</p>)
	supervisor(supervisor)
	researcher(researcher)
	writer(writer)
	qa(qa)
	synthesize(synthesize)
	__end__(<p>__end__</p>)
	__start__ --> supervisor;
	supervisor --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 12) What to remember

- The supervisor decides the work plan.
- `Send` fans work out dynamically.
- Worker outputs merge back through a reducer.
- `operator.add` is a simple reducer for list accumulation.
- The same pattern scales to more agents or longer workflows.


## Key takeaways

LangGraph’s multi-agent patterns are built around orchestration, dynamic fan-out, and reducer-based state merging. The docs show both supervisor-style workflows and Send API map-reduce patterns, which is why they fit neatly together in this notebook. 


## References

- Multi-agent: https://docs.langchain.com/oss/python/langgraph/multi-agent
- Multi-agent collaboration: https://docs.langchain.com/oss/python/langgraph/multi-agent-collaboration
- Run graph nodes in parallel: https://docs.langchain.com/oss/python/langgraph/use-graph-api#run-graph-nodes-in-parallel
- Reducers: https://docs.langchain.com/oss/python/langgraph/graph-api#reducers
